# Trash Classifier — Colab Runner (PyTorch)

Runner buat ngejalanin paket `trash-classifier` di Colab dengan GPU gratis.

**Langkah:**
1. `Runtime -> Change runtime type -> GPU`
2. Jalankan sel berurutan dari atas.
3. Sel 2: upload `trash-classifier.zip` (zip seluruh folder project).
4. Sel 3: upload `dataset-resized.zip`.
5. Sel 4: latih + banding 3 model + ensemble.
6. Sel 5: Grad-CAM heatmaps. Sel 6: simpan hasil (zip+download).

In [ ]:
# 1. Cek GPU (torch & torchvision sudah ada di Colab)
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2. Upload kode project (zip folder trash-classifier lalu pilih di sini)
from google.colab import files
import zipfile, os
up = files.upload()                       # pilih trash-classifier.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('.')
# pindah ke root project (folder yang berisi src/ dan configs/)
for root, dirs, fs in os.walk('.'):
    if 'configs' in dirs and 'src' in dirs:
        os.chdir(root); break
print('Project dir:', os.getcwd())
!pip install -q -r requirements.txt

In [ ]:
# 3. Upload dataset (dataset-resized.zip -> ekstrak ke data/)
from google.colab import files
import zipfile, os
os.makedirs('data', exist_ok=True)
up = files.upload()                       # pilih dataset-resized.zip
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall('data')
for c in ['cardboard','glass','metal','paper','plastic','trash']:
    p = f'data/dataset-resized/{c}'
    print(f'  {c:10s}: {len(os.listdir(p)) if os.path.isdir(p) else "MISSING"}')

In [ ]:
# 4. Latih + banding 3 model + ensemble
!python scripts/compare.py --configs configs/resnet50.yaml configs/efficientnet_b0.yaml configs/mobilenet_v2.yaml

In [ ]:
# 5. Grad-CAM heatmaps (explainability) untuk tiap model
for arch in ['resnet50', 'efficientnet_b0', 'mobilenet_v2']:
    !python scripts/gradcam.py --config configs/{arch}.yaml
from IPython.display import Image as IPImage, display
print('=== ResNet50 Grad-CAM ===')
display(IPImage('outputs/resnet50/gradcam.png'))

In [ ]:
# 6. Simpan semua hasil (model + metrics + gradcam) -> zip + download
import shutil
from google.colab import files
shutil.make_archive('trash_outputs', 'zip', 'outputs')
files.download('trash_outputs.zip')